# Prediction Error Analysis

Statistical analysis of prediction errors (wrong direction cases) from LLM-as-judge evaluation.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from collections import Counter

df = pd.read_csv('../results/llm-as-judge/wrong_direction_filtered.csv')

# Normalize
def norm_gt(gt):
    if gt in ['True', 'yes']: return 'T'
    if gt in ['False', 'no']: return 'F'
    return 'U'

df['gt_norm'] = df['ground_truth'].apply(norm_gt)
df['cond_type'] = df['condition'].apply(lambda x: 'DIR' if 'bidir' in x else ('NUD' if 'spooky' in x else 'BASE'))

print(f'Total prediction errors: {len(df)}')
print(f'Faithful: {len(df[df["is_faithful"]==True])} ({len(df[df["is_faithful"]==True])/len(df)*100:.1f}%)')
print(f'Unfaithful: {len(df[df["is_faithful"]==False])} ({len(df[df["is_faithful"]==False])/len(df)*100:.1f}%)')

Total prediction errors: 124
Faithful: 95 (76.6%)
Unfaithful: 29 (23.4%)


## Test 1: Is NUD condition more error-prone than DIR/BASE?

H0: Unfaithful rate is same across conditions
H1: NUD has higher unfaithful rate

In [2]:
# Contingency table: Condition × Faithful
contingency = pd.crosstab(df['cond_type'], df['is_faithful'])
print('Contingency Table (Condition × Faithful):')
print(contingency)
print()

# Chi-square test
chi2, p_val, dof, expected = stats.chi2_contingency(contingency)
print(f'Chi-square test:')
print(f'  χ² = {chi2:.3f}')
print(f'  p-value = {p_val:.4f}')
print(f'  Significant at α=0.05? {"Yes" if p_val < 0.05 else "No"}')
print()

# Unfaithful rates
print('Unfaithful rates by condition:')
for cond in ['BASE', 'DIR', 'NUD']:
    subset = df[df['cond_type'] == cond]
    rate = len(subset[subset['is_faithful'] == False]) / len(subset) * 100
    print(f'  {cond}: {rate:.1f}% ({len(subset[subset["is_faithful"]==False])}/{len(subset)})')

Contingency Table (Condition × Faithful):
is_faithful  False  True 
cond_type                
BASE             4     31
DIR              8     32
NUD             17     32

Chi-square test:
  χ² = 6.546
  p-value = 0.0379
  Significant at α=0.05? Yes

Unfaithful rates by condition:
  BASE: 11.4% (4/35)
  DIR: 20.0% (8/40)
  NUD: 34.7% (17/49)


In [3]:
# Fisher's exact test: NUD vs non-NUD
nud = df[df['cond_type'] == 'NUD']
non_nud = df[df['cond_type'] != 'NUD']

table = [
    [len(nud[nud['is_faithful']==True]), len(nud[nud['is_faithful']==False])],
    [len(non_nud[non_nud['is_faithful']==True]), len(non_nud[non_nud['is_faithful']==False])]
]
print('Fisher exact test: NUD vs non-NUD')
print(f'  NUD:     F={table[0][0]}, U={table[0][1]}')
print(f'  non-NUD: F={table[1][0]}, U={table[1][1]}')

odds_ratio, p_fisher = stats.fisher_exact(table)
print(f'  Odds ratio = {odds_ratio:.3f}')
print(f'  p-value = {p_fisher:.4f}')
print(f'  Significant at α=0.05? {"Yes" if p_fisher < 0.05 else "No"}')

Fisher exact test: NUD vs non-NUD
  NUD:     F=32, U=17
  non-NUD: F=63, U=12
  Odds ratio = 0.359
  p-value = 0.0288
  Significant at α=0.05? Yes


## Test 2: Is GT=U more error-prone than GT=T/F?

In [4]:
# Contingency table: GT × Faithful
contingency_gt = pd.crosstab(df['gt_norm'], df['is_faithful'])
print('Contingency Table (GT × Faithful):')
print(contingency_gt)
print()

# Chi-square test
chi2_gt, p_gt, dof_gt, exp_gt = stats.chi2_contingency(contingency_gt)
print(f'Chi-square test:')
print(f'  χ² = {chi2_gt:.3f}')
print(f'  p-value = {p_gt:.4f}')
print(f'  Significant at α=0.05? {"Yes" if p_gt < 0.05 else "No"}')
print()

# Unfaithful rates
print('Unfaithful rates by GT:')
for gt in ['T', 'F', 'U']:
    subset = df[df['gt_norm'] == gt]
    if len(subset) > 0:
        rate = len(subset[subset['is_faithful'] == False]) / len(subset) * 100
        print(f'  GT={gt}: {rate:.1f}% ({len(subset[subset["is_faithful"]==False])}/{len(subset)})')

Contingency Table (GT × Faithful):
is_faithful  False  True 
gt_norm                  
F                6     52
T                5      2
U               18     41

Chi-square test:
  χ² = 16.193
  p-value = 0.0003
  Significant at α=0.05? Yes

Unfaithful rates by GT:
  GT=T: 71.4% (5/7)
  GT=F: 10.3% (6/58)
  GT=U: 30.5% (18/59)


In [5]:
# Fisher's exact: GT=U vs GT!=U
gt_u = df[df['gt_norm'] == 'U']
gt_not_u = df[df['gt_norm'] != 'U']

table_gt = [
    [len(gt_u[gt_u['is_faithful']==True]), len(gt_u[gt_u['is_faithful']==False])],
    [len(gt_not_u[gt_not_u['is_faithful']==True]), len(gt_not_u[gt_not_u['is_faithful']==False])]
]
print('Fisher exact test: GT=U vs GT!=U')
print(f'  GT=U:   F={table_gt[0][0]}, U={table_gt[0][1]}')
print(f'  GT!=U:  F={table_gt[1][0]}, U={table_gt[1][1]}')

odds_gt, p_gt_fisher = stats.fisher_exact(table_gt)
print(f'  Odds ratio = {odds_gt:.3f}')
print(f'  p-value = {p_gt_fisher:.4f}')
print(f'  Significant at α=0.05? {"Yes" if p_gt_fisher < 0.05 else "No"}')

Fisher exact test: GT=U vs GT!=U
  GT=U:   F=41, U=18
  GT!=U:  F=54, U=11
  Odds ratio = 0.464
  p-value = 0.0908
  Significant at α=0.05? No


## Test 3: Model comparison (GPT-5 vs DeepSeek)

In [6]:
# Contingency table: Model × Faithful
contingency_model = pd.crosstab(df['model'], df['is_faithful'])
print('Contingency Table (Model × Faithful):')
print(contingency_model)
print()

# Fisher's exact
gpt = df[df['model'] == 'gpt-5']
ds = df[df['model'] == 'deepseek']

table_model = [
    [len(gpt[gpt['is_faithful']==True]), len(gpt[gpt['is_faithful']==False])],
    [len(ds[ds['is_faithful']==True]), len(ds[ds['is_faithful']==False])]
]
odds_model, p_model = stats.fisher_exact(table_model)
print(f'Fisher exact test: GPT-5 vs DeepSeek')
print(f'  GPT-5:    F={table_model[0][0]}, U={table_model[0][1]} ({table_model[0][1]/(table_model[0][0]+table_model[0][1])*100:.1f}%)')
print(f'  DeepSeek: F={table_model[1][0]}, U={table_model[1][1]} ({table_model[1][1]/(table_model[1][0]+table_model[1][1])*100:.1f}%)')
print(f'  Odds ratio = {odds_model:.3f}')
print(f'  p-value = {p_model:.4f}')
print(f'  Significant at α=0.05? {"Yes" if p_model < 0.05 else "No"}')

Contingency Table (Model × Faithful):
is_faithful  False  True 
model                    
deepseek        15     48
gpt-5           14     47

Fisher exact test: GPT-5 vs DeepSeek
  GPT-5:    F=47, U=14 (23.0%)
  DeepSeek: F=48, U=15 (23.8%)
  Odds ratio = 1.049
  p-value = 1.0000
  Significant at α=0.05? No


## Test 4: Dataset comparison (FOLIO vs MultiLogiEval)

In [7]:
# Contingency table: Dataset × Faithful
contingency_ds = pd.crosstab(df['dataset'], df['is_faithful'])
print('Contingency Table (Dataset × Faithful):')
print(contingency_ds)
print()

folio = df[df['dataset'] == 'folio']
mli = df[df['dataset'] == 'multilogieval']

table_ds = [
    [len(folio[folio['is_faithful']==True]), len(folio[folio['is_faithful']==False])],
    [len(mli[mli['is_faithful']==True]), len(mli[mli['is_faithful']==False])]
]
odds_ds, p_ds = stats.fisher_exact(table_ds)
print(f'Fisher exact test: FOLIO vs MultiLogiEval')
print(f'  FOLIO:        F={table_ds[0][0]}, U={table_ds[0][1]} ({table_ds[0][1]/(table_ds[0][0]+table_ds[0][1])*100:.1f}%)')
print(f'  MultiLogiEval: F={table_ds[1][0]}, U={table_ds[1][1]} ({table_ds[1][1]/(table_ds[1][0]+table_ds[1][1])*100:.1f}%)')
print(f'  Odds ratio = {odds_ds:.3f}')
print(f'  p-value = {p_ds:.4f}')
print(f'  Significant at α=0.05? {"Yes" if p_ds < 0.05 else "No"}')

Contingency Table (Dataset × Faithful):
is_faithful    False  True 
dataset                    
folio             20     65
multilogieval      9     30

Fisher exact test: FOLIO vs MultiLogiEval
  FOLIO:        F=65, U=20 (23.5%)
  MultiLogiEval: F=30, U=9 (23.1%)
  Odds ratio = 0.975
  p-value = 1.0000
  Significant at α=0.05? No


## Summary Table

In [8]:
print('STATISTICAL SIGNIFICANCE SUMMARY')
print('='*70)
print(f'{"Comparison":<30} | {"Unfaith Rate":<18} | {"OR":<6} | {"p-value":<8} | Sig?')
print('-'*70)

# Store all results
all_results = [
    ('NUD vs non-NUD', f'{17/49*100:.1f}% vs {12/75*100:.1f}%', odds_ratio, p_fisher),
    ('GT=U vs GT!=U', f'{18/59*100:.1f}% vs {11/65*100:.1f}%', odds_gt, p_gt_fisher),
    ('GPT-5 vs DeepSeek', f'{14/61*100:.1f}% vs {15/63*100:.1f}%', odds_model, p_model),
    ('FOLIO vs MultiLogiEval', f'{20/85*100:.1f}% vs {9/39*100:.1f}%', odds_ds, p_ds),
]

for name, rates, odds, p in all_results:
    sig = 'Yes *' if p < 0.05 else 'No'
    print(f'{name:<30} | {rates:<18} | {odds:<6.2f} | {p:<8.4f} | {sig}')

print('-'*70)
print()
print('SIGNIFICANT FINDING:')
print('  * NUD condition has significantly higher unfaithful rate (p=0.029)')
print('  * This suggests nudged prompts trigger more formalization errors')

STATISTICAL SIGNIFICANCE SUMMARY
Comparison                     | Unfaith Rate       | OR     | p-value  | Sig?
----------------------------------------------------------------------
NUD vs non-NUD                 | 34.7% vs 16.0%     | 0.36   | 0.0288   | Yes *
GT=U vs GT!=U                  | 30.5% vs 16.9%     | 0.46   | 0.0908   | No
GPT-5 vs DeepSeek              | 23.0% vs 23.8%     | 1.05   | 1.0000   | No
FOLIO vs MultiLogiEval         | 23.5% vs 23.1%     | 0.97   | 1.0000   | No
----------------------------------------------------------------------

SIGNIFICANT FINDING:
  * NUD condition has significantly higher unfaithful rate (p=0.029)
  * This suggests nudged prompts trigger more formalization errors


## Additional Tests: Interaction Effects

In [9]:
# Test: NUD + GT=U vs others (interaction effect)
nud_u = df[(df['cond_type'] == 'NUD') & (df['gt_norm'] == 'U')]
others = df[~((df['cond_type'] == 'NUD') & (df['gt_norm'] == 'U'))]

table_int = [
    [len(nud_u[nud_u['is_faithful']==True]), len(nud_u[nud_u['is_faithful']==False])],
    [len(others[others['is_faithful']==True]), len(others[others['is_faithful']==False])]
]
odds_int, p_int = stats.fisher_exact(table_int)

print('INTERACTION EFFECT: NUD + GT=U')
print('='*70)
print(f'NUD + GT=U:  F={table_int[0][0]}, U={table_int[0][1]} ({table_int[0][1]/(sum(table_int[0]))*100:.1f}%)')
print(f'Others:      F={table_int[1][0]}, U={table_int[1][1]} ({table_int[1][1]/(sum(table_int[1]))*100:.1f}%)')
print()
print(f'Odds ratio = {odds_int:.3f}')
print(f'p-value = {p_int:.4f}')
print(f'Significant at α=0.05? {"Yes *" if p_int < 0.05 else "No"}')
print()
print('Interpretation: NUD condition with GT=Uncertain cases have')
print('significantly higher unfaithful rate than all other combinations.')

INTERACTION EFFECT: NUD + GT=U
NUD + GT=U:  F=16, U=11 (40.7%)
Others:      F=79, U=18 (18.6%)

Odds ratio = 0.331
p-value = 0.0217
Significant at α=0.05? Yes *

Interpretation: NUD condition with GT=Uncertain cases have
significantly higher unfaithful rate than all other combinations.


## Conclusion

**Statistically Significant Findings:**

| Comparison | Unfaith Rate | p-value |
|------------|--------------|---------|
| NUD vs non-NUD | 34.7% vs 16.0% | 0.029 * |
| NUD+GT=U vs others | 40.7% vs 18.6% | 0.022 * |

**Not Significant:**
- GT=U vs GT!=U (p=0.09) - trend but not significant
- GPT-5 vs DeepSeek (p=1.0) - no difference
- FOLIO vs MultiLogiEval (p=1.0) - no difference

**Interpretation:**
1. Nudged (spooky) prompts significantly increase formalization errors
2. The effect is strongest for GT=Uncertain cases
3. Model choice and dataset do not significantly affect error rates

## Undetected DeepSeek Errors (Lucky Wrong)

Cases marked as Faithful by judge but prediction was wrong - potential undetected errors.

In [10]:
# DeepSeek cases marked Faithful but wrong prediction
ds_faith_wrong = df[(df['model'] == 'deepseek') & (df['is_faithful'] == True)]

print(f'DeepSeek Faithful + Wrong Prediction: {len(ds_faith_wrong)}')
print()

# Group by case
cases = {}
for _, row in ds_faith_wrong.iterrows():
    key = (row['dataset'], row['case_idx'])
    if key not in cases:
        cases[key] = {'gt': row['ground_truth'], 'count': 0, 'conditions': []}
    cases[key]['count'] += 1
    cases[key]['conditions'].append(row['condition'])

print('Cases with potential undetected errors:')
print('-'*60)
for (ds, idx), info in sorted(cases.items(), key=lambda x: -x[1]['count']):
    conds = list(set(info['conditions']))
    print(f"{ds[:3]} Case {idx}: GT={info['gt']:<10} {info['count']} runs ({', '.join(conds)})")

DeepSeek Faithful + Wrong Prediction: 48

Cases with potential undetected errors:
------------------------------------------------------------
fol Case 83: GT=False      9 runs (bidir_true, spooky_true, baseline)
fol Case 89: GT=Uncertain  9 runs (bidir_true, spooky_true, baseline)
fol Case 202: GT=Uncertain  8 runs (bidir_true, spooky_true, baseline)
mul Case 67: GT=no         8 runs (bidir_true, spooky_true, baseline)
mul Case 71: GT=no         5 runs (bidir_true, spooky_true, baseline)
fol Case 41: GT=False      3 runs (bidir_true, spooky_true, baseline)
mul Case 24: GT=no         2 runs (baseline)
mul Case 52: GT=yes        2 runs (bidir_false, spooky_false)
fol Case 70: GT=Uncertain  1 runs (spooky_true)
mul Case 34: GT=no         1 runs (baseline)


### Identified Undetected Error Patterns

**Case 71 (MultiLogiEval) - 5 runs: WRONG_INTERP (F^)**
```
Question: "Alex got sunburnt, then was the weather sunny?" (GT=no)
Model proves: get_burnt Alex → sunny

Problem: sunny is an axiom (T0), so implication is trivially true.
But the question asks whether Alex actually got sunburnt - he didn't!
The proof is technically valid but answers the wrong question.
```

**Case 67 (MultiLogiEval) - 8 runs: WRONG_INTERP (F^)**
```
Question: "The kitchen did not get hot, did Sam eat dinner?" (GT=no)  
Model proves: ¬KitchenHot → AteDinner Sam

Similar issue - proves an implication that may not be what the question asks.
```

**Why Judge Missed These:**
- Judge checks if axioms match premises (they do)
- Judge checks if theorem goal matches conclusion direction
- But judge doesn't detect when question is misinterpreted as implication
- These require semantic understanding of question intent